# ARIMA/SARIMA Experiment

Walmart Store Sales Forecasting — ARIMA experiment

**MLflow Experiment:** `ARIMA_Training`

---

In [1]:
import sys
sys.path.insert(0, '/Users/r00t/Claude/Projects/ML final project')
from utils.feature_engineering import wmae

import os
import time
import json
import pickle
import tempfile
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, kpss, acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import seasonal_decompose
import pmdarima as pm
from pmdarima.model_selection import RollingForecastCV
from collections import Counter
from joblib import Parallel, delayed

import dagshub
dagshub.init(repo_owner='tgela23', repo_name='walmart-sales-forecasting', mlflow=True)

import mlflow
import mlflow.pyfunc
mlflow.set_experiment('ARIMA_Training')

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (11, 5)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

N_JOBS = os.cpu_count() or 4
# The spec targets up to 200 (Store, Dept) series for the CV / Best_Model runs, but auto_arima
# with a seasonal m=52 stepwise search takes ~3 min/series on this machine -- 200 series would
# take hours even parallelized across all cores. N_SERIES_SAMPLE is the single knob to scale
# back up to 200 given more time/compute; everything else in the pipeline is unchanged.
N_SERIES_SAMPLE = 25

Accessing as tgela23

Initialized MLflow to track repo "tgela23/walmart-sales-forecasting"

Repository tgela23/walmart-sales-forecasting initialized!

## 1. Data Loading

In [2]:
DATA_DIR = '/Users/r00t/Claude/Projects/ML final project/data/raw/walmart-recruiting-store-sales-forecasting/'
train = pd.read_csv(DATA_DIR + 'train.csv', parse_dates=['Date'])
stores = pd.read_csv(DATA_DIR + 'stores.csv')
features = pd.read_csv(DATA_DIR + 'features.csv', parse_dates=['Date'])
test = pd.read_csv(DATA_DIR + 'test.csv', parse_dates=['Date'])

print(f'train: {train.shape} | stores: {stores.shape} | features: {features.shape} | test: {test.shape}')
display(train.head())

n_stores = train['Store'].nunique()
n_depts = train['Dept'].nunique()
n_series = train.groupby(['Store', 'Dept']).ngroups

print(f'Date range: {train["Date"].min().date()} -> {train["Date"].max().date()}')
print(f'n_stores={n_stores}, n_depts={n_depts}, n_unique_series={n_series}, total_rows={len(train)}')

train: (421570, 5) | stores: (45, 3) | features: (8190, 12) | test: (115064, 4)


,Store,Dept,Date,Weekly_Sales,IsHoliday
0,1,1,2010-02-05,24924.50,False
1,1,1,2010-02-12,46039.49,True
2,1,1,2010-02-19,41595.55,False
3,1,1,2010-02-26,19403.54,False
4,1,1,2010-03-05,21827.90,False


Date range: 2010-02-05 -> 2012-10-26
n_stores=45, n_depts=81, n_unique_series=3331, total_rows=421570


## 2. Preprocessing & Feature Engineering

*MLflow run: `ARIMA_Cleaning`*

In [3]:
with mlflow.start_run(run_name='ARIMA_Cleaning') as run_cleaning:
    print('run_id:', run_cleaning.info.run_id)

    MD_COLS = ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']

    merged = train.merge(stores, on='Store', how='left')
    merged = merged.merge(features, on=['Store', 'Date'], how='left', suffixes=('', '_feat'))
    if 'IsHoliday_feat' in merged.columns:
        merged = merged.drop(columns=['IsHoliday_feat'])

    n_rows_raw = len(merged)

    markdown_null_pct_before_fill = float(merged[MD_COLS].isna().mean().mean() * 100)
    merged[MD_COLS] = merged[MD_COLS].fillna(0)

    n_negative_sales = int((merged['Weekly_Sales'] < 0).sum())
    merged['Weekly_Sales'] = merged['Weekly_Sales'].clip(lower=0)

    df_clean = merged.sort_values(['Store', 'Dept', 'Date']).reset_index(drop=True)
    n_rows_after_cleaning = len(df_clean)

    n_stores = int(df_clean['Store'].nunique())
    n_depts = int(df_clean['Dept'].nunique())
    n_unique_series = int(df_clean.groupby(['Store', 'Dept']).ngroups)
    holiday_week_pct = float(df_clean['IsHoliday'].mean() * 100)

    date_range_start = str(df_clean['Date'].min().date())
    date_range_end = str(df_clean['Date'].max().date())

    mlflow.log_param('n_rows_raw', n_rows_raw)
    mlflow.log_param('n_rows_after_cleaning', n_rows_after_cleaning)
    mlflow.log_param('n_stores', n_stores)
    mlflow.log_param('n_depts', n_depts)
    mlflow.log_param('n_unique_series', n_unique_series)
    mlflow.log_param('date_range_start', date_range_start)
    mlflow.log_param('date_range_end', date_range_end)
    mlflow.log_metric('markdown_null_pct_before_fill', markdown_null_pct_before_fill)
    mlflow.log_metric('n_negative_sales', n_negative_sales)
    mlflow.log_metric('holiday_week_pct', holiday_week_pct)

    # Time-based split: last 8 weeks as holdout validation (mirrors the other model notebooks)
    unique_dates = np.sort(df_clean['Date'].unique())
    val_dates = unique_dates[-8:]
    train_dates = unique_dates[:-8]

    train_portion = df_clean[df_clean['Date'].isin(train_dates)].reset_index(drop=True)
    val_holdout = df_clean[df_clean['Date'].isin(val_dates)].reset_index(drop=True)

    val_holdout_start = str(pd.Timestamp(val_dates[0]).date())
    val_holdout_end = str(pd.Timestamp(val_dates[-1]).date())
    n_train_rows = len(train_portion)
    n_val_rows = len(val_holdout)

    mlflow.log_param('val_holdout_start', val_holdout_start)
    mlflow.log_param('val_holdout_end', val_holdout_end)
    mlflow.log_metric('n_train_rows', n_train_rows)
    mlflow.log_metric('n_val_rows', n_val_rows)

    summary = pd.DataFrame({
        'metric': ['n_rows_raw', 'n_rows_after_cleaning', 'n_stores', 'n_depts', 'n_unique_series',
                   'markdown_null_pct_before_fill', 'n_negative_sales', 'holiday_week_pct',
                   'date_range_start', 'date_range_end',
                   'val_holdout_start', 'val_holdout_end', 'n_train_rows', 'n_val_rows'],
        'value': [n_rows_raw, n_rows_after_cleaning, n_stores, n_depts, n_unique_series,
                  f'{markdown_null_pct_before_fill:.2f}%', n_negative_sales, f'{holiday_week_pct:.2f}%',
                  date_range_start, date_range_end,
                  val_holdout_start, val_holdout_end, n_train_rows, n_val_rows],
    })
    print(summary.to_string(index=False))

run_id: c19595965a8f4865b5110da1c4552767


                       metric      value
                   n_rows_raw     421570
        n_rows_after_cleaning     421570
                     n_stores         45
                      n_depts         81
              n_unique_series       3331
markdown_null_pct_before_fill     67.48%
             n_negative_sales       1285
             holiday_week_pct      7.04%
             date_range_start 2010-02-05
               date_range_end 2012-10-26
            val_holdout_start 2012-09-07
              val_holdout_end 2012-10-26
                 n_train_rows     397841
                   n_val_rows      23729


🏃 View run ARIMA_Cleaning at: https://dagshub.com/tgela23/walmart-sales-forecasting.mlflow/#/experiments/2/runs/c19595965a8f4865b5110da1c4552767
🧪 View experiment at: https://dagshub.com/tgela23/walmart-sales-forecasting.mlflow/#/experiments/2


## 3. Feature Selection

*MLflow run: `ARIMA_Feature_Selection`*

In [4]:
with mlflow.start_run(run_name='ARIMA_Feature_Selection') as run_fs:
    print('run_id:', run_fs.info.run_id)

    # ARIMA is a per-series model -- "feature selection" here means stationarity analysis
    # and order identification rather than picking columns.
    all_pairs = list(df_clean.groupby(['Store', 'Dept']).groups.keys())
    rng = np.random.RandomState(RANDOM_STATE)
    n_sample = min(50, len(all_pairs))
    sample_idx = rng.choice(len(all_pairs), size=n_sample, replace=False)
    sample_pairs = [all_pairs[i] for i in sample_idx]

    def get_series(store, dept):
        return (df_clean[(df_clean['Store'] == store) & (df_clean['Dept'] == dept)]
                .sort_values('Date')['Weekly_Sales'].values.astype(float))

    stat_results = []
    for store, dept in sample_pairs:
        s = get_series(store, dept)
        if len(s) < 10:
            continue
        try:
            adf_p = float(adfuller(s, autolag='AIC')[1])
        except Exception:
            adf_p = np.nan
        try:
            kpss_p = float(kpss(s, regression='c', nlags='auto')[1])
        except Exception:
            kpss_p = np.nan
        stat_results.append({
            'store': store, 'dept': dept, 'length': len(s),
            'adf_p': adf_p, 'is_stationary_adf': (adf_p < 0.05) if pd.notna(adf_p) else np.nan,
            'kpss_p': kpss_p, 'is_stationary_kpss': (kpss_p >= 0.05) if pd.notna(kpss_p) else np.nan,
        })

    stat_df = pd.DataFrame(stat_results)
    n_series_tested = len(stat_df)

    valid_adf = stat_df['is_stationary_adf'].dropna()
    valid_kpss = stat_df['is_stationary_kpss'].dropna()
    valid_both = stat_df.dropna(subset=['is_stationary_adf', 'is_stationary_kpss'])

    pct_adf_stationary = float(valid_adf.mean() * 100) if len(valid_adf) else 0.0
    pct_kpss_stationary = float(valid_kpss.mean() * 100) if len(valid_kpss) else 0.0
    pct_both_stationary = (
        float((valid_both['is_stationary_adf'] & valid_both['is_stationary_kpss']).mean() * 100)
        if len(valid_both) else 0.0
    )
    d_recommended = 0 if pct_adf_stationary > 60 else 1

    mlflow.log_param('n_series_tested', n_series_tested)
    mlflow.log_metric('pct_adf_stationary', pct_adf_stationary)
    mlflow.log_metric('pct_kpss_stationary', pct_kpss_stationary)
    mlflow.log_metric('pct_both_stationary', pct_both_stationary)
    mlflow.log_param('d_recommended', d_recommended)

    print(f'Series tested: {n_series_tested}')
    print(f'ADF stationary: {pct_adf_stationary:.1f}% | KPSS stationary: {pct_kpss_stationary:.1f}% | '
          f'Both agree: {pct_both_stationary:.1f}%')
    print(f'Recommended d: {d_recommended}')

    # --- ACF/PACF for 3 representative series: high / medium / low volume ---
    # Restricted to series with enough history (>=20 rows) -- several series clip to a total
    # of exactly 0 after the negative-sales clip and can be as short as 1 row, which breaks acf().
    series_lengths_all = df_clean.groupby(['Store', 'Dept']).size()
    long_enough = series_lengths_all[series_lengths_all >= 20].index
    series_totals = (
        df_clean.groupby(['Store', 'Dept'])['Weekly_Sales'].sum()
        .loc[long_enough]
        .sort_values(ascending=False)
    )
    high_key = series_totals.index[0]
    mid_key = series_totals.index[len(series_totals) // 2]
    low_key = series_totals.index[-1]
    rep_keys = [('High-volume', high_key), ('Medium-volume', mid_key), ('Low-volume', low_key)]

    fig, axes = plt.subplots(3, 2, figsize=(12, 12))
    for row, (label, (store, dept)) in enumerate(rep_keys):
        s = get_series(store, dept)
        lags = max(1, min(40, len(s) // 2 - 1))
        plot_acf(s, ax=axes[row, 0], lags=lags, title=f'{label} ACF — Store {store} Dept {dept}')
        plot_pacf(s, ax=axes[row, 1], lags=lags, title=f'{label} PACF — Store {store} Dept {dept}')
    plt.tight_layout()

    acf_pacf_path = os.path.join(tempfile.gettempdir(), 'acf_pacf_sample.png')
    fig.savefig(acf_pacf_path, dpi=100, bbox_inches='tight')
    plt.close(fig)
    mlflow.log_artifact(acf_pacf_path)
    os.remove(acf_pacf_path)

    # --- Seasonal decomposition (period=52) on the top store-dept series ---
    top_store, top_dept = high_key
    top_subset = (df_clean[(df_clean['Store'] == top_store) & (df_clean['Dept'] == top_dept)]
                  .sort_values('Date').reset_index(drop=True))
    top_ts = pd.Series(top_subset['Weekly_Sales'].values, index=pd.DatetimeIndex(top_subset['Date']))
    decomposition = seasonal_decompose(top_ts, model='additive', period=52, extrapolate_trend='freq')

    fig2, axes2 = plt.subplots(4, 1, figsize=(11, 9), sharex=True)
    axes2[0].plot(decomposition.observed); axes2[0].set_title(f'Observed — Store {top_store} Dept {top_dept}')
    axes2[1].plot(decomposition.trend); axes2[1].set_title('Trend')
    axes2[2].plot(decomposition.seasonal); axes2[2].set_title('Seasonality (period=52)')
    axes2[3].plot(decomposition.resid); axes2[3].set_title('Residual')
    plt.tight_layout()

    decomp_path = os.path.join(tempfile.gettempdir(), 'seasonal_decomposition.png')
    fig2.savefig(decomp_path, dpi=100, bbox_inches='tight')
    plt.close(fig2)
    mlflow.log_artifact(decomp_path)
    os.remove(decomp_path)

    # --- Series-length stats across the full cleaned dataset ---
    all_lengths = df_clean.groupby(['Store', 'Dept']).size()
    avg_series_length = float(all_lengths.mean())
    min_series_length = int(all_lengths.min())
    max_series_length = int(all_lengths.max())
    full_history_len = int(df_clean['Date'].nunique())
    pct_series_with_full_history = float((all_lengths == full_history_len).mean() * 100)

    mlflow.log_metric('avg_series_length', avg_series_length)
    mlflow.log_metric('min_series_length', min_series_length)
    mlflow.log_metric('max_series_length', max_series_length)
    mlflow.log_metric('pct_series_with_full_history', pct_series_with_full_history)

    print(f'Series length — avg: {avg_series_length:.1f}, min: {min_series_length}, max: {max_series_length}')
    print(f'% series with full {full_history_len}-week history: {pct_series_with_full_history:.1f}%')

    order_note = (
        f'ADF/KPSS results recommend d={d_recommended} (non-seasonal differencing). '
        f'ACF/PACF inspection across high/medium/low-volume series and the seasonal '
        f'decomposition (strong 52-week retail cycle) motivate a seasonal component of '
        f'P<=1, D=1, Q<=1 at m=52, with non-seasonal (p,q) in the 0-3 range per the '
        f'ACF/PACF cutoffs of the sampled series.'
    )
    mlflow.log_param('recommended_order_range', order_note)
    print(order_note)

run_id: ab63a52cc7a440e8aabb567e51314dfc


/var/folders/9x/86nr0qg15hs7gwb4kg0_dtp00000gn/T/ipykernel_9103/1378639625.py:26: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  kpss_p = float(kpss(s, regression='c', nlags='auto')[1])
/var/folders/9x/86nr0qg15hs7gwb4kg0_dtp00000gn/T/ipykernel_9103/1378639625.py:26: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.

  kpss_p = float(kpss(s, regression='c', nlags='auto')[1])
/var/folders/9x/86nr0qg15hs7gwb4kg0_dtp00000gn/T/ipykernel_9103/1378639625.py:26: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  kpss_p = float(kpss(s, regression='c', nlags='auto')[1])
/var/folders/9x/86nr0qg15hs7gwb4kg0_dtp00000gn/T/ipykernel_9103/1378639625.p

/var/folders/9x/86nr0qg15hs7gwb4kg0_dtp00000gn/T/ipykernel_9103/1378639625.py:26: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  kpss_p = float(kpss(s, regression='c', nlags='auto')[1])
/var/folders/9x/86nr0qg15hs7gwb4kg0_dtp00000gn/T/ipykernel_9103/1378639625.py:26: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  kpss_p = float(kpss(s, regression='c', nlags='auto')[1])
/var/folders/9x/86nr0qg15hs7gwb4kg0_dtp00000gn/T/ipykernel_9103/1378639625.py:26: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.

  kpss_p = float(kpss(s, regression='c', nlags='auto')[1])
/var/folders/9x/86nr0qg15hs7gwb4kg0_dtp00000gn/T/ipykernel_9103/1378639625.p

Series tested: 47
ADF stationary: 74.5% | KPSS stationary: 70.2% | Both agree: 59.6%
Recommended d: 0


Series length — avg: 126.6, min: 1, max: 143
% series with full 143-week history: 79.9%


ADF/KPSS results recommend d=0 (non-seasonal differencing). ACF/PACF inspection across high/medium/low-volume series and the seasonal decomposition (strong 52-week retail cycle) motivate a seasonal component of P<=1, D=1, Q<=1 at m=52, with non-seasonal (p,q) in the 0-3 range per the ACF/PACF cutoffs of the sampled series.


🏃 View run ARIMA_Feature_Selection at: https://dagshub.com/tgela23/walmart-sales-forecasting.mlflow/#/experiments/2/runs/ab63a52cc7a440e8aabb567e51314dfc
🧪 View experiment at: https://dagshub.com/tgela23/walmart-sales-forecasting.mlflow/#/experiments/2


## 4. Model Training & Cross-Validation

*MLflow run: `ARIMA_CV`*

In [5]:
with mlflow.start_run(run_name='ARIMA_CV') as run_cv:
    print('run_id:', run_cv.info.run_id)

    mlflow.log_param('model', 'ARIMA')
    mlflow.log_param('n_jobs', N_JOBS)

    # auto_arima config -- the ARIMA equivalents of n_estimators/max_depth for tree models
    mlflow.log_param('seasonal', True)
    mlflow.log_param('m', 52)
    mlflow.log_param('stepwise', True)
    mlflow.log_param('max_p', 3)
    mlflow.log_param('max_q', 3)
    mlflow.log_param('max_P', 1)
    mlflow.log_param('max_Q', 1)
    mlflow.log_param('d', 'auto (None)')
    mlflow.log_param('D', 1)
    mlflow.log_param('information_criterion', 'aic')
    mlflow.log_param('out_of_sample_size', 8)
    mlflow.log_param('scoring', 'mse')

    # Sample: top 20 stores by total sales x top 10 depts by row count, capped at N_SERIES_SAMPLE
    # (spec targets up to 200 -- see the N_SERIES_SAMPLE note in cell 1 for why it's reduced here).
    top_stores = df_clean.groupby('Store')['Weekly_Sales'].sum().nlargest(20).index.tolist()
    top_depts = df_clean.groupby('Dept')['Weekly_Sales'].count().nlargest(10).index.tolist()

    series_sizes = df_clean.groupby(['Store', 'Dept']).size()
    candidate_pairs = [
        (s, d) for s in top_stores for d in top_depts
        if series_sizes.get((s, d), 0) > 20
    ]
    sampled_pairs = candidate_pairs[:N_SERIES_SAMPLE]
    n_series_sampled = len(sampled_pairs)
    mlflow.log_param('n_series_sampled', n_series_sampled)
    print(f'Running CV on {n_series_sampled} (Store, Dept) pairs '
          f'(of {len(candidate_pairs)} candidates; {N_JOBS} parallel workers)...')

    def fit_one_cv(store, dept):
        subset = (df_clean[(df_clean['Store'] == store) & (df_clean['Dept'] == dept)]
                  .sort_values('Date').reset_index(drop=True))
        train_part, val_part = subset.iloc[:-8], subset.iloc[-8:]
        train_series = train_part['Weekly_Sales'].values.astype(float)
        val_series = val_part['Weekly_Sales'].values.astype(float)
        is_holiday_val = val_part['IsHoliday'].astype(int).values
        try:
            model = pm.auto_arima(
                train_series,
                seasonal=True, m=52, stepwise=True,
                suppress_warnings=True, error_action='ignore',
                max_p=3, max_q=3, max_P=1, max_Q=1,
                d=None, D=1, information_criterion='aic',
                out_of_sample_size=8, scoring='mse',
            )
            preds = np.clip(model.predict(n_periods=8), 0, None)
            score = wmae(val_series, preds, is_holiday_val)
            p, d_, q = model.order
            P, D, Q, _ = model.seasonal_order
            return {'store': store, 'dept': dept, 'wmae': score, 'p': p, 'd': d_, 'q': q,
                    'P': P, 'D': D, 'Q': Q, 'aic': model.aic(), 'error': None}
        except Exception as e:
            return {'store': store, 'dept': dept, 'wmae': np.nan, 'p': None, 'd': None, 'q': None,
                    'P': None, 'D': None, 'Q': None, 'aic': np.nan, 'error': str(e)}

    # Parallelized across N_JOBS worker processes (joblib) -- with these settings a single
    # series takes ~3 min sequentially, so per-series progress prints aren't meaningful here;
    # joblib's own progress logging (verbose=5) reports batch completions instead.
    t0 = time.time()
    cv_results = Parallel(n_jobs=N_JOBS, verbose=5)(
        delayed(fit_one_cv)(store, dept) for store, dept in sampled_pairs
    )
    print(f'\nCV fitting completed in {time.time() - t0:.1f}s for {n_series_sampled} series')

    cv_df = pd.DataFrame(cv_results)
    n_series_failed = int(cv_df['error'].notna().sum())
    valid = cv_df[cv_df['error'].isna()].copy()

    mean_cv_wmae = float(valid['wmae'].mean())
    std_cv_wmae = float(valid['wmae'].std())
    median_cv_wmae = float(valid['wmae'].median())
    min_cv_wmae = float(valid['wmae'].min())
    max_cv_wmae = float(valid['wmae'].max())
    pct_series_under_2000_wmae = float((valid['wmae'] < 2000).mean() * 100)
    mean_aic = float(valid['aic'].mean())

    most_common_p = int(Counter(valid['p']).most_common(1)[0][0])
    most_common_d = int(Counter(valid['d']).most_common(1)[0][0])
    most_common_q = int(Counter(valid['q']).most_common(1)[0][0])
    most_common_P = int(Counter(valid['P']).most_common(1)[0][0])
    most_common_D = int(Counter(valid['D']).most_common(1)[0][0])
    most_common_Q = int(Counter(valid['Q']).most_common(1)[0][0])
    most_common_order = f'({most_common_p},{most_common_d},{most_common_q})'

    mlflow.log_metric('mean_cv_wmae', mean_cv_wmae)
    mlflow.log_metric('std_cv_wmae', std_cv_wmae)
    mlflow.log_metric('median_cv_wmae', median_cv_wmae)
    mlflow.log_metric('min_cv_wmae', min_cv_wmae)
    mlflow.log_metric('max_cv_wmae', max_cv_wmae)
    mlflow.log_metric('pct_series_under_2000_wmae', pct_series_under_2000_wmae)
    mlflow.log_metric('mean_aic', mean_aic)
    mlflow.log_metric('n_series_failed', n_series_failed)
    mlflow.log_param('most_common_p', most_common_p)
    mlflow.log_param('most_common_d', most_common_d)
    mlflow.log_param('most_common_q', most_common_q)
    mlflow.log_param('most_common_P', most_common_P)
    mlflow.log_param('most_common_D', most_common_D)
    mlflow.log_param('most_common_Q', most_common_Q)
    mlflow.log_param('most_common_order', most_common_order)

    print(f'\nCV WMAE — mean: {mean_cv_wmae:.2f}, std: {std_cv_wmae:.2f}, median: {median_cv_wmae:.2f}, '
          f'min: {min_cv_wmae:.2f}, max: {max_cv_wmae:.2f}')
    print(f'% series WMAE < 2000: {pct_series_under_2000_wmae:.1f}% | mean AIC: {mean_aic:.1f} | '
          f'failed: {n_series_failed}')
    print(f'Most common order: {most_common_order} seasonal ({most_common_P},{most_common_D},{most_common_Q},52)')

    # Bar chart of per-series WMAE distribution
    fig, ax = plt.subplots(figsize=(10, 5))
    sorted_wmae = valid.sort_values('wmae')['wmae'].values
    ax.bar(range(len(sorted_wmae)), sorted_wmae, color=sns.color_palette('deep')[0])
    ax.axhline(mean_cv_wmae, color='black', linestyle='--', label=f'mean = {mean_cv_wmae:.0f}')
    ax.set_xlabel('Series (sorted)')
    ax.set_ylabel('WMAE')
    ax.set_title('Per-series CV WMAE distribution')
    ax.legend()
    plt.tight_layout()

    dist_path = os.path.join(tempfile.gettempdir(), 'cv_wmae_distribution.png')
    fig.savefig(dist_path, dpi=100, bbox_inches='tight')
    plt.close(fig)
    mlflow.log_artifact(dist_path)
    os.remove(dist_path)

    top5_worst_series = valid.nlargest(5, 'wmae').apply(
        lambda r: f"{int(r['store'])}_{int(r['dept'])}", axis=1).tolist()
    top5_best_series = valid.nsmallest(5, 'wmae').apply(
        lambda r: f"{int(r['store'])}_{int(r['dept'])}", axis=1).tolist()
    mlflow.log_param('top5_worst_series', json.dumps(top5_worst_series))
    mlflow.log_param('top5_best_series', json.dumps(top5_best_series))

    print(f'Top 5 worst series: {top5_worst_series}')
    print(f'Top 5 best series: {top5_best_series}')

run_id: 28b34cadf3014e818316842da07ea8e4


Running CV on 25 (Store, Dept) pairs (of 200 candidates; 8 parallel workers)...


[Parallel(n_jobs=8)]: Using backend LokyBackend with 8 concurrent workers.


[Parallel(n_jobs=8)]: Done   2 tasks      | elapsed:  3.2min


[Parallel(n_jobs=8)]: Done  16 out of  25 | elapsed:  9.0min remaining:  5.1min


...

[Parallel(n_jobs=8)]: Done  25 out of  25 | elapsed: 15.6min finished



CV fitting completed in 937.4s for 25 series



CV WMAE — mean: 3669.27, std: 3194.35, median: 2631.88, min: 568.23, max: 12963.90
% series WMAE < 2000: 40.0% | mean AIC: 1659.8 | failed: 0
Most common order: (0,0,0) seasonal (0,1,0,52)


Top 5 worst series: ['14_2', '14_1', '20_7', '14_4', '14_7']
Top 5 best series: ['20_3', '4_3', '4_14', '20_16', '20_4']


🏃 View run ARIMA_CV at: https://dagshub.com/tgela23/walmart-sales-forecasting.mlflow/#/experiments/2/runs/28b34cadf3014e818316842da07ea8e4
🧪 View experiment at: https://dagshub.com/tgela23/walmart-sales-forecasting.mlflow/#/experiments/2


## 5. Best Model — Save as Pipeline

*Saved to MLflow Model Registry*

In [6]:
class ARIMAWrapper(mlflow.pyfunc.PythonModel):
    """Stores per-series (order, seasonal_order, trend, fitted params, history) rather than
    the full pmdarima/statsmodels result objects -- a single fitted SARIMAX with m=52 pickles
    to ~250MB (it retains the full Kalman filter/smoother state arrays), which made the
    25-series artifact too large to upload (DagsHub's gateway returned HTTP 413). The
    lightweight package is ~1-2KB/series and reconstructs an identical model via
    `SARIMAX(history, order=..., seasonal_order=...).filter(params)`, which applies the
    already-fitted parameters (no re-optimization) and reproduces forecasts exactly.
    """

    def load_context(self, context):
        with open(context.artifacts['models'], 'rb') as f:
            self.models = pickle.load(f)

    def predict(self, context, model_input):
        # model_input: DataFrame with Store, Dept, n_periods columns
        results = []
        for _, row in model_input.iterrows():
            key = (int(row['Store']), int(row['Dept']))
            n = int(row.get('n_periods', 8))
            pkg = self.models.get(key)
            if pkg is None:
                preds = [np.nan] * n
            else:
                sm_mod = sm.tsa.SARIMAX(
                    pkg['history'], order=pkg['order'], seasonal_order=pkg['seasonal_order'],
                    trend=pkg['trend'],
                )
                sm_res = sm_mod.filter(pkg['params'])
                preds = np.clip(sm_res.get_forecast(steps=n).predicted_mean, 0, None).tolist()
            results.append({'Store': row['Store'], 'Dept': row['Dept'], 'forecast': preds})
        return pd.DataFrame(results)


with mlflow.start_run(run_name='ARIMA_Best_Model') as run_best:
    print('run_id:', run_best.info.run_id)

    mlflow.log_param('model', 'ARIMA')
    mlflow.log_param('seasonal', True)
    mlflow.log_param('m', 52)
    mlflow.log_param('stepwise', True)
    mlflow.log_param('max_p', 3)
    mlflow.log_param('max_q', 3)
    mlflow.log_param('max_P', 1)
    mlflow.log_param('max_Q', 1)
    mlflow.log_param('d', 'auto (None)')
    mlflow.log_param('D', 1)
    mlflow.log_param('information_criterion', 'aic')
    mlflow.log_param('n_series_sampled', len(sampled_pairs))

    def fit_one_final(store, dept):
        subset = (df_clean[(df_clean['Store'] == store) & (df_clean['Dept'] == dept)]
                  .sort_values('Date').reset_index(drop=True))
        full_series = subset['Weekly_Sales'].values.astype(float)
        try:
            model = pm.auto_arima(
                full_series,
                seasonal=True, m=52, stepwise=True,
                suppress_warnings=True, error_action='ignore',
                max_p=3, max_q=3, max_P=1, max_Q=1,
                d=None, D=1, information_criterion='aic',
            )
            in_sample_preds = np.clip(np.asarray(model.predict_in_sample())[-8:], 0, None)
            last8_true = full_series[-8:]
            last8_holiday = subset['IsHoliday'].astype(int).values[-8:]
            score = wmae(last8_true, in_sample_preds, last8_holiday)

            # Lightweight package instead of the full model object -- see ARIMAWrapper docstring.
            sm_model = model.arima_res_.model
            package = {
                'order': sm_model.order,
                'seasonal_order': sm_model.seasonal_order,
                'trend': sm_model.trend,
                'params': np.asarray(model.arima_res_.params),
                'history': full_series,
            }
            return store, dept, package, score, None
        except Exception as e:
            return store, dept, None, np.nan, str(e)

    print(f'Fitting final models on {len(sampled_pairs)} series (full training data, no holdout)...')
    t0 = time.time()
    results = Parallel(n_jobs=N_JOBS, verbose=5)(
        delayed(fit_one_final)(store, dept) for store, dept in sampled_pairs
    )
    print(f'\nFinal fitting completed in {time.time() - t0:.1f}s')

    fitted_models = {}
    final_scores = []
    n_failed = 0
    for store, dept, package, score, err in results:
        if package is not None:
            fitted_models[(store, dept)] = package
            final_scores.append(score)
        else:
            n_failed += 1

    n_models_fitted = len(fitted_models)
    final_train_wmae = float(np.mean(final_scores)) if final_scores else np.nan
    final_train_wmae_std = float(np.std(final_scores)) if final_scores else np.nan

    mlflow.log_metric('n_models_fitted', n_models_fitted)
    mlflow.log_metric('n_failed', n_failed)
    mlflow.log_metric('final_train_wmae', final_train_wmae)
    mlflow.log_metric('final_train_wmae_std', final_train_wmae_std)

    print(f'Fitted {n_models_fitted} models ({n_failed} failed)')
    print(f'Final in-sample WMAE (last 8 weeks): {final_train_wmae:.2f} +/- {final_train_wmae_std:.2f}')

    # Save fitted orders summary as a JSON artifact
    orders_summary = {
        f'{store}_{dept}': {'order': list(pkg['order']), 'seasonal_order': list(pkg['seasonal_order'])}
        for (store, dept), pkg in fitted_models.items()
    }
    orders_json_path = os.path.join(tempfile.gettempdir(), 'orders_summary.json')
    with open(orders_json_path, 'w') as f:
        json.dump(orders_summary, f, indent=2)
    mlflow.log_artifact(orders_json_path)
    os.remove(orders_json_path)

    # Pickle the lightweight fitted-package dict and log the pyfunc wrapper referencing it
    models_pickle_path = os.path.join(tempfile.gettempdir(), 'arima_models.pkl')
    with open(models_pickle_path, 'wb') as f:
        pickle.dump(fitted_models, f)
    print(f'Pickled models artifact size: {os.path.getsize(models_pickle_path) / 1e6:.2f} MB')

    input_example = pd.DataFrame([
        {'Store': sampled_pairs[0][0], 'Dept': sampled_pairs[0][1], 'n_periods': 8},
    ])
    mlflow.pyfunc.log_model(
        name='ARIMA_pipeline',
        python_model=ARIMAWrapper(),
        artifacts={'models': models_pickle_path},
        registered_model_name='ARIMA_Pipeline',
        input_example=input_example,
    )
    os.remove(models_pickle_path)

    print(f'\nModel registered as ARIMA_Pipeline (run_id={run_best.info.run_id})')

    # --- Sanity check: reload from the registry and predict on 3 sample rows ---
    client = mlflow.MlflowClient()
    versions = client.search_model_versions("name='ARIMA_Pipeline'")
    latest_version = max(int(v.version) for v in versions)

    loaded_model = mlflow.pyfunc.load_model(f'models:/ARIMA_Pipeline/{latest_version}')
    sample_rows = pd.DataFrame([
        {'Store': s, 'Dept': d, 'n_periods': 8} for s, d in list(fitted_models.keys())[:3]
    ])
    sample_preds = loaded_model.predict(sample_rows)
    print(f'\nLoaded ARIMA_Pipeline version {latest_version}')
    print(sample_preds)

    print('\n=== Summary ===')
    print(f'n_series fitted     : {n_models_fitted}')
    print(f'mean_cv_wmae        : {mean_cv_wmae:.2f}')
    print(f'final_train_wmae    : {final_train_wmae:.2f}')
    print(f'most_common_order   : {most_common_order} seasonal ({most_common_P},{most_common_D},{most_common_Q},52)')

/Users/r00t/Claude/Projects/ML final project/.venv/lib/python3.12/site-packages/mlflow/pyfunc/utils/data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


run_id: aef7a9aafa2f4a1cb575b6bc18917191


Fitting final models on 25 series (full training data, no holdout)...


[Parallel(n_jobs=8)]: Using backend LokyBackend with 8 concurrent workers.


[Parallel(n_jobs=8)]: Done   2 tasks      | elapsed:  1.5min


...

Created version '2' of model 'ARIMA_Pipeline'.


Python(10065) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(10066) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.



Model registered as ARIMA_Pipeline (run_id=aef7a9aafa2f4a1cb575b6bc18917191)



Loaded ARIMA_Pipeline version 2
   Store  Dept                                           forecast
0     20     1  [43017.46168151983, 30703.782768921083, 34032....
1     20     2  [79894.7957142857, 77949.44571428571, 74131.48...
2     20     3  [10984.622092409605, 11944.056365202749, 12061...

=== Summary ===
n_series fitted     : 25
mean_cv_wmae        : 3669.27
final_train_wmae    : 2579.07
most_common_order   : (0,0,0) seasonal (0,1,0,52)


🏃 View run ARIMA_Best_Model at: https://dagshub.com/tgela23/walmart-sales-forecasting.mlflow/#/experiments/2/runs/aef7a9aafa2f4a1cb575b6bc18917191
🧪 View experiment at: https://dagshub.com/tgela23/walmart-sales-forecasting.mlflow/#/experiments/2
